# 03 - Injection gallery: 15 types x {naive, consistent}

Companion notebook to `results/figures/injection_*.png`. It runs the seeded
injector once and plots one example per type, showing that **consistent** edits
keep the electrical identities while **naive** single-channel edits break them
(hypothesis H1).

Prerequisites: `python -m energymind.io` and `python -m energymind.clean`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from energymind import clean, inject, inject_plots as G, io
from energymind.config import load_settings

warnings.filterwarnings("ignore", category=FutureWarning)
%config InlineBackend.figure_format = "png"
plt.rcParams["figure.dpi"] = 110

PREVALENCE, SEED = 0.05, 1
settings = load_settings()
baseline = clean.load_baseline(settings)
injected, labels = inject.inject_all(baseline, settings, PREVALENCE, SEED)
FIG = io.resolve("results/figures")
print(f"baseline {baseline.shape}, injected {injected.shape}, {len(labels)} events -> {FIG}")

## Labels at a glance (computed)

In [ ]:
summary = (
    labels.groupby(["origin", "type", "variant"])
    .agg(events=("anomaly_id", "size"), median_severity=("severity", lambda s: s.mode().iat[0]))
    .reset_index()
)
summary.style.hide(axis="index")

## Overview

Events by type and origin, the severity mix (effect size in sigma units), the
variant and scope split.

In [ ]:
G.fig_injection_overview(labels, PREVALENCE, FIG)
plt.show()

## Active power: one example per type x variant

Baseline in grey, injected in the origin colour, event window shaded.

In [ ]:
G.fig_type_gallery(baseline, injected, labels, FIG)
plt.show()

## Identity residual: naive breaks, consistent holds

The maximum absolute electrical-identity residual per interval (log scale), with
the 1e-2 tolerance. Consistent edits sit below the tolerance; naive edits shoot
far above it.

In [ ]:
G.fig_residual_gallery(baseline, injected, labels, FIG)
plt.show()

## Detailed comparison for selected types

Active power, voltage and the identity residual for naive vs consistent.

In [ ]:
for name in ["additive_bias", "gain_error", "voltage_drift", "noise_burst", "stuck_at"]:
    if name in set(labels["type"]):
        G.fig_variant_comparison(baseline, injected, labels, name, FIG)
        plt.show()

## Label schema

One row per injected event, with everything needed for event-level evaluation.

In [ ]:
labels.head(10)